# TextSwinUMamba — Colab Training

Trains `TextSwinUMambaD` on ISIC 2017 with GPT-4o captions as text input.

**Prerequisites:** run `colab_data_prep.ipynb` once to verify the dataset and
precompute BERT text features. This notebook assumes all Drive artifacts are ready:

```
MyDrive/TextSwinUMamba/
├── datasets/isic2017/{train,val}/{images,masks}/
├── captions/captions.jsonl
├── pretrained/vmamba_tiny_e292.pth
├── cache/text_features.pt          ← generated by colab_data_prep.ipynb
├── env_config.sh                   ← written by step 2 below
└── runs/                           ← checkpoints + TensorBoard logs land here
```

**Resume:** if Colab disconnects mid-training, re-run top to bottom. The Miniconda
install, conda env, and wheel installs are all idempotent. Training picks up from
`last.pth` on Drive.

## 1. Configure remote + mount Drive

In [ ]:
TEXTSWINUMAMBA_REPO = 'https://github.com/<you>/TextSwinUMamba.git'
TEXTSWINUMAMBA_BRANCH = 'main'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Write env_config.sh to Drive

Every `%%bash` cell below sources this script first. It activates the conda env,
exports Drive paths, sets MPLBACKEND, and `cd`s into the repo. Surviving across
disconnects is automatic — the script lives on Drive.

In [ ]:
%%writefile /content/drive/MyDrive/TextSwinUMamba/env_config.sh
#!/bin/bash

# 1. Activate conda env
source /usr/local/miniconda/bin/activate textswinumamba

# 2. Master paths (Drive for data + artifacts, /content for code repo)
export WORK_ROOT="/content/drive/MyDrive/TextSwinUMamba"
export REPO_DIR="/content/TextSwinUMamba"

export ISIC_ROOT="$WORK_ROOT/datasets/isic2017"
export CAPTIONS="$WORK_ROOT/captions/captions.jsonl"
export PRETRAINED="$WORK_ROOT/pretrained/vmamba_tiny_e292.pth"
export TEXT_CACHE="$WORK_ROOT/cache/text_features.pt"
export RUNS_DIR="$WORK_ROOT/runs"

# 3. Common crash workarounds
export MPLBACKEND="Agg"

# 4. Auto-navigate into the repo
cd "$REPO_DIR" 2>/dev/null || true

## 3. Install Miniconda + create conda env + install all wheels

This cell is **idempotent** — re-runs skip work that's already done.
First-time runtime: ~5 min. Subsequent runs: ~30 sec.

The two GitHub wheel URLs are the key magic: they're pre-compiled for exactly
Python 3.10 + CUDA 11.8 + torch 2.0 + cxx11abi=FALSE (= Colab's base image).

In [ ]:
%%bash

# 1. Install Miniconda only if missing
if [ ! -d "/usr/local/miniconda" ]; then
    echo "Downloading Miniconda..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
    bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda
fi
export PATH="/usr/local/miniconda/bin:$PATH"

# Accept Anaconda TOS (newer conda versions require this)
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r    || true

# 2. Create conda env if missing
conda create -n textswinumamba python=3.10 -y || true
source activate textswinumamba

echo "Installing PyTorch (CUDA 11.8)..."
pip install -q torch==2.0.1+cu118 torchvision==0.15.2+cu118 \
    --index-url https://download.pytorch.org/whl/cu118

echo "Installing causal-conv1d + mamba-ssm from GitHub wheels (no source build)..."
pip install -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.1.1/causal_conv1d-1.1.1+cu118torch2.0cxx11abiFALSE-cp310-cp310-linux_x86_64.whl
pip install -q https://github.com/state-spaces/mamba/releases/download/v1.1.1/mamba_ssm-1.1.1+cu118torch2.0cxx11abiFALSE-cp310-cp310-linux_x86_64.whl

echo "Installing TextSwinUMamba dependencies..."
pip install -q einops timm numba torchinfo packaging ninja
pip install -q albumentations tensorboard matplotlib pyyaml tqdm
# Compatibility pins — must come AFTER the rest to override anything pip pulled in.
pip install -q "numpy<2.0" "transformers==4.36.2" "opencv-python==4.8.1.78"

echo "Done. Activate with: source /usr/local/miniconda/bin/activate textswinumamba"

## 4. Verify the mamba-ssm import path the model uses

In [ ]:
%%bash
source /content/drive/MyDrive/TextSwinUMamba/env_config.sh
python -c "from mamba_ssm.ops.selective_scan_interface import selective_scan_fn, selective_scan_ref; print('mamba-ssm OK')"
python -c "import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda)"

## 5. Clone the repo

Repo lives in `/content/TextSwinUMamba`, fresh each session. Drive holds the
data + artifacts, not the code.

In [ ]:
import os
os.environ['TEXTSWINUMAMBA_REPO']   = TEXTSWINUMAMBA_REPO
os.environ['TEXTSWINUMAMBA_BRANCH'] = TEXTSWINUMAMBA_BRANCH

In [ ]:
%%bash
source /content/drive/MyDrive/TextSwinUMamba/env_config.sh
rm -rf "$REPO_DIR"
git clone --branch "$TEXTSWINUMAMBA_BRANCH" "$TEXTSWINUMAMBA_REPO" "$REPO_DIR"
ls "$REPO_DIR"

## 6. Assert data is ready

All four artifacts must show `[OK]`. If `TEXT_CACHE` is missing, run
`colab_data_prep.ipynb` first.

In [ ]:
%%bash
source /content/drive/MyDrive/TextSwinUMamba/env_config.sh
mkdir -p "$WORK_ROOT/cache" "$RUNS_DIR"
all_ok=true
for label_path in "ISIC_ROOT:$ISIC_ROOT" "CAPTIONS:$CAPTIONS" "PRETRAINED:$PRETRAINED" "TEXT_CACHE:$TEXT_CACHE"; do
    label="${label_path%%:*}"; path="${label_path#*:}"
    if [ -e "$path" ]; then status="OK"; else status="MISSING"; all_ok=false; fi
    printf "%-12s %s  [%s]\n" "$label" "$path" "$status"
done
if [ "$all_ok" = false ]; then
    echo ""
    echo "ERROR: Missing artifacts. Run colab_data_prep.ipynb first."
    exit 1
fi
echo ""
echo "All artifacts present — ready to train."

## 7. Patch config to point at Drive paths

Writes a Colab-flavored copy alongside the canonical config — git stays clean.

In [ ]:
%%bash
source /content/drive/MyDrive/TextSwinUMamba/env_config.sh
python - <<'PY'
import os, yaml
src = os.path.join(os.environ['REPO_DIR'], 'configs/isic2017.yaml')
dst = os.path.join(os.environ['REPO_DIR'], 'configs/isic2017_colab.yaml')
with open(src) as f: cfg = yaml.safe_load(f)
cfg['data']['isic_root']           = os.environ['ISIC_ROOT']
cfg['data']['captions_jsonl']      = os.environ['CAPTIONS']
cfg['data']['text_features_cache'] = os.environ['TEXT_CACHE']
cfg['model']['pretrained_ckpt']    = os.environ['PRETRAINED']
cfg['output']['base_dir']          = os.environ['RUNS_DIR']
cfg['output']['tensorboard']       = True
cfg['train']['max_hours']          = 11.5
with open(dst, 'w') as f: yaml.safe_dump(cfg, f, sort_keys=False)
print('wrote', dst)
print('  run_dir will be:', os.path.join(os.environ['RUNS_DIR'], cfg['run_name']))
PY

## 8. Launch TensorBoard

Points at the Drive runs folder; survives across disconnects.

In [ ]:
RUNS_DIR = '/content/drive/MyDrive/TextSwinUMamba/runs'
%load_ext tensorboard
%tensorboard --logdir {RUNS_DIR} --port 6006

## 9. Train (auto-resumes from `last.pth` on Drive)

If the runtime disconnects mid-training, re-run the notebook top to bottom.
All setup steps are idempotent. Training picks up at the next epoch with
optimizer / scheduler / RNG state restored.

In [ ]:
%%bash
source /content/drive/MyDrive/TextSwinUMamba/env_config.sh
python train.py --config configs/isic2017_colab.yaml --resume auto

## 10. Evaluate the best checkpoint

In [ ]:
%%bash
source /content/drive/MyDrive/TextSwinUMamba/env_config.sh
RUN_NAME=$(python -c "import yaml; print(yaml.safe_load(open('configs/isic2017_colab.yaml'))['run_name'])")
BEST_CKPT="$RUNS_DIR/$RUN_NAME/best.pth"
echo "Using checkpoint: $BEST_CKPT"
python evaluate.py --config configs/isic2017_colab.yaml --ckpt "$BEST_CKPT"